# Ideal MHD stability scan with ActorGPEC

This notebook demonstrates how to use `FUSE.StudyDatabaseGenerator` together with `ActorGPEC` to scan a single equilibrium parameter and observe how the ideal-MHD stability energy δW(n=1) changes.

**What we do:**
1. Load the D3D `:default` base case with boundary built from scalar parameters.
2. Vary the major radius R0 over ±12 % in 11 uniform steps using the `↔` notation.
3. Run `ActorGPEC` (n=1, no-wall vacuum) at every point using `StudyDatabaseGenerator` (serial execution).
4. Extract δW from `dd.mhd_linear` and plot it against R0.

**Expected result:** a smooth curve — noisy/jumping values would signal a bug in the IMAS read path or the equilibrium solve.

In [ ]:
using FUSE
import IMAS
import JSON
using Plots
using Printf
FUSE.logging(Logging.Info; actors=Logging.Error);

## 1. Base case and GPEC settings

In [ ]:
ini, act = FUSE.case_parameters(:D3D, :default)

# Build the boundary from scalar parameters (R0, a, κ, δ) so that
# varying ini.equilibrium.R0 actually changes the equilibrium geometry.
ini.equilibrium.boundary_from = :scalars

# GPEC: n=1 toroidal mode, no conducting wall, vacuum perturbation only
act.ActorGPEC.nn_low   = 1
act.ActorGPEC.nn_high  = 1
act.ActorGPEC.vac_flag = true
act.ActorGPEC.wall_shape = "nowall"
act.ActorGPEC.verbose  = false
act.ActorGPEC.psihigh  = 0.95   # stay inside the FUSE equilibrium domain
act.ActorGPEC.qlow     = 0.0    # no q threshold

R0_base = ini.equilibrium.R0
println("Base major radius: R0 = $(round(R0_base; sigdigits=4)) m")

## 2. Parameter scan definition

The `↔` operator marks a parameter as a scan variable.  
`StudyDatabaseGenerator` samples `n_simulations` points uniformly between the two bounds.

In [ ]:
# Vary R0 ±12 % around the nominal value in 11 uniform steps
R0_min = 0.88 * R0_base
R0_max = 1.12 * R0_base
ini.equilibrium.R0 = R0_base ↔ [R0_min, R0_max]

println("Scan range: R0 ∈ [$(round(R0_min; sigdigits=4)), $(round(R0_max; sigdigits=4))] m")

## 3. Study configuration and workflow

In [ ]:
sty = FUSE.study_parameters(:DatabaseGenerator)
sty.n_workers      = 0              # 0 = serial execution on the local machine
sty.n_simulations  = 11
sty.save_folder    = mktempdir()    # temporary directory, change to a persistent path if desired
sty.file_save_mode = :overwrite
sty.database_policy = :separate_folders

println("Results will be saved to: $(sty.save_folder)")

In [ ]:
# Workflow: initialise the equilibrium, then run ActorGPEC
function workflow_DatabaseGenerator(dd, ini, act)
    FUSE.init(dd, ini, act)
    FUSE.ActorGPEC(dd, act)
    return nothing
end

study = FUSE.StudyDatabaseGenerator(sty, ini, act)
study.workflow = workflow_DatabaseGenerator

## 4. Run the scan

In [ ]:
FUSE.run(study)

## 5. Extract δW from saved results

In [ ]:
R0_values = Float64[]
dW_values = Float64[]

for (k, folder) in enumerate(sort(filter(isdir, readdir(sty.save_folder; join=true))))
    dd_file = joinpath(folder, "dd.json")
    !isfile(dd_file) && (println("  Case $k: dd.json not found"); continue)
    try
        # Load only equilibrium + mhd_linear to avoid type-conversion issues
        # from unrelated FUSE fields present in the full dd.json.
        d = open(JSON.parse, dd_file)
        case_dd = IMAS.dd()
        IMAS.dict2imas(d["equilibrium"], case_dd.equilibrium)
        IMAS.dict2imas(d["mhd_linear"],  case_dd.mhd_linear)

        R0 = case_dd.equilibrium.time_slice[1].global_quantities.magnetic_axis.r

        if length(case_dd.mhd_linear.time_slice) > 0 &&
           length(case_dd.mhd_linear.time_slice[1].toroidal_mode) > 0
            δW = real(case_dd.mhd_linear.time_slice[1].toroidal_mode[1].energy_perturbed)
            push!(R0_values, Float64(R0))
            push!(dW_values, Float64(δW))
            @printf "  Case %2d: R0 = %.3f m  →  δW(n=1) = %+.5f\n" k R0 δW
        else
            println("  Case $k: no mhd_linear output (GPEC may have failed)")
        end
    catch e
        println("  Case $k: failed to load — $(sprint(showerror, e))")
    end
end

println("\n$(length(dW_values))/$(sty.n_simulations) points succeeded.")

## 6. Plot δW vs R0

In [ ]:
length(dW_values) < 2 && error("Too few successful points to plot — check ActorGPEC setup.")

# Sort by R0 for a clean line plot
perm      = sortperm(R0_values)
R0_sorted = R0_values[perm]
dW_sorted = dW_values[perm]

p = plot(
    R0_sorted, dW_sorted;
    xlabel     = "Major radius  R0  [m]",
    ylabel     = "δW(n=1)  [normalized]",
    title      = "Ideal MHD stability vs major radius\nD3D :default  |  n=1  |  no-wall",
    marker     = :circle,
    markersize = 5,
    lw         = 2,
    label      = "δW(n=1)",
    legend     = :topright,
)

hline!(p, [0.0];
    linestyle = :dash,
    color     = :red,
    lw        = 1.5,
    label     = "marginal stability  (δW = 0)",
)

vline!(p, [R0_base];
    linestyle = :dot,
    color     = :gray,
    lw        = 1.5,
    label     = "nominal R0 = $(round(R0_base; sigdigits=4)) m",
)

display(p)

**Reading the plot:**
- A smooth curve confirms that the IMAS read path and ActorGPEC are working correctly end-to-end.
- A noisy or jumping curve would indicate a problem in the spline construction inside `read_imas`.
- Where the curve crosses δW = 0 is the marginal-stability radius for the n=1 kink mode without a conducting wall.